# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.


**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. 
- The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

- Your first mission is to familiarize yourself with the **Books to Scrape** website. 
- Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

**Next**, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

- After reviewing the site, you can construct a plan for scraping relevant data. 
- Pay attention to the details displayed for each book, including the title, price, rating, and availability. 
- This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

- In the fast-changing online world, websites often update and change their structures. 
    - When you try this lab, the **Books to Scrape** website might differ from what you expect.

- If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

- You can choose another website that interests you and is suitable for scraping data. 
- Options like Wikipedia, The New York Times, or even library databases are great alternatives. 
- The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. 
- This is your opportunity to practice and adapt to different web environments!

---
# Solved Lab | Web Scraping
---

In [2]:
import requests
url = "https://books.toscrape.com/index.html"
response = requests.get(url)
response

<Response [200]>

In [4]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.content, 'html.parser')

In [5]:
soup.find_all('p', attrs={'class': 'star-rating'})

[<p class="star-rating Three">
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 </p>,
 <p class="star-rating One">
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 </p>,
 <p class="star-rating One">
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 </p>,
 <p class="star-rating Four">
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 </p>,
 <p class="star-rating Five">
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 </p>,
 <p class="star-rating One">
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i 

In [28]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import time
import re

# Function to extract book details from individual book page
def extract_book_details(book_url):
    details = {
        "UPC": None,
        "Description": None,
        "Genre": None,
        "Availability (Amount)": None,
        "Number of Reviews": None
    }
    
    try:
        response = requests.get(book_url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # UPC - look for table row with UPC
        table_rows = soup.find_all("tr")
        for tr in table_rows:
            th = tr.find("th")
            td = tr.find("td")
            if th and td:
                th_text = th.get_text(strip=True)
                if th_text == "UPC":
                    details["UPC"] = td.get_text(strip=True)
                elif th_text == "Number of reviews":
                    details["Number of Reviews"] = td.get_text(strip=True)
        
        # Availability amount - extract number from "In stock (20 available)"
        avail_elem = soup.find("p", class_="instock availability")
        if avail_elem:
            avail_text = avail_elem.get_text(strip=True)
            amount_match = re.search(r'\((\d+) available\)', avail_text)
            if amount_match:
                details["Availability (Amount)"] = amount_match.group(1)
        
        # Description - get first <p> after #product_description
        desc_div = soup.find("div", id="product_description")
        if desc_div:
            desc_p = desc_div.find_next_sibling("p")
            if desc_p:
                details["Description"] = desc_p.get_text(strip=True)
        
        # Genre - third breadcrumb link (index 2)
        breadcrumb_links = soup.select("ul.breadcrumb li a")
        if len(breadcrumb_links) >= 3:
            details["Genre"] = breadcrumb_links[2].get_text(strip=True)
    
    except requests.RequestException as e:
        print(f"Failed to fetch {book_url}: {e}")
    
    # Sleep a bit to be polite
    time.sleep(0.5)
    
    return details

# Load the CSV
books_df = pd.read_csv("books.csv")

all_details = []

# Loop through every book with progress bar
for idx, row in tqdm(books_df.iterrows(), total=len(books_df), desc="Scraping books"):
    book_url = row['Link']
    
    # Convert relative URL to absolute if necessary
    if book_url.startswith("catalogue/"):
        book_url = "https://books.toscrape.com/" + book_url
    
    details = extract_book_details(book_url)
    all_details.append(details)

# Combine original CSV with new details
details_df = pd.DataFrame(all_details)
final_df = pd.concat([books_df, details_df], axis=1)

# Save the full dataset
final_df.to_csv("books_complete_data.csv", index=False)
print("Saved all book details to books_detailed.csv")


Scraping books:   1%|▏         | 1/75 [00:01<01:30,  1.23s/it]

Failed to fetch https://books.toscrape.com/the-four-agreements-a-practical-guide-to-personal-freedom_970/index.html: 404 Client Error: Not Found for url: https://books.toscrape.com/the-four-agreements-a-practical-guide-to-personal-freedom_970/index.html


Scraping books:   3%|▎         | 2/75 [00:02<01:17,  1.07s/it]

Failed to fetch https://books.toscrape.com/sophies-world_966/index.html: 404 Client Error: Not Found for url: https://books.toscrape.com/sophies-world_966/index.html


Scraping books:   4%|▍         | 3/75 [00:03<01:12,  1.01s/it]

...

Failed to fetch https://books.toscrape.com/the-edge-of-reason-bridget-jones-2_63/index.html: 404 Client Error: Not Found for url: https://books.toscrape.com/the-edge-of-reason-bridget-jones-2_63/index.html


Scraping books:  99%|█████████▊| 74/75 [01:11<00:00,  1.03it/s]

Failed to fetch https://books.toscrape.com/a-spys-devotion-the-regency-spies-of-london-1_3/index.html: 404 Client Error: Not Found for url: https://books.toscrape.com/a-spys-devotion-the-regency-spies-of-london-1_3/index.html


Scraping books: 100%|██████████| 75/75 [01:12<00:00,  1.03it/s]

Saved all book details to books_detailed.csv


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm  # For a nice progress bar
import time

# Function to extract book details from individual book page
def extract_book_details(book_url):
    details = {
        "UPC": None,
        "Description": None,
        "Genre": None,
        "Availability (Amount)": None,
        "Number of Reviews": None
    }
    
    try:
        response = requests.get(book_url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # UPC
        upc_elem = soup.select_one("th:contains('UPC') + td")
        if upc_elem:
            details["UPC"] = upc_elem.text.strip()
        
        # Availability amount
        avail_elem = soup.select_one("th:contains('Availability') + td")
        if avail_elem:
            details["Availability (Amount)"] = avail_elem.text.strip()
        
        # Number of reviews
        reviews_elem = soup.select_one("th:contains('Number of reviews') + td")
        if reviews_elem:
            details["Number of Reviews"] = reviews_elem.text.strip()
        
        # Description
        desc_elem = soup.select_one("#product_description")
        if desc_elem:
            # Description is in the next <p> tag after the h2
            desc_paragraph = desc_elem.find_next_sibling("p")
            if desc_paragraph:
                details["Description"] = desc_paragraph.text.strip()
        
        # Genre
        breadcrumb = soup.select("ul.breadcrumb li a")
        if len(breadcrumb) >= 3:
            details["Genre"] = breadcrumb[2].text.strip()
    
    except requests.RequestException as e:
        print(f"Failed to fetch {book_url}: {e}")
    
    # Sleep a bit to be polite
    time.sleep(0.5)
    
    return details

# Load the CSV
books_df = pd.read_csv("books.csv")

all_details = []

# Loop through every book with progress bar
for idx, row in tqdm(books_df.iterrows(), total=len(books_df), desc="Scraping books"):
    book_url = row['Link']
    
    # Convert relative URL to absolute if necessary
    if book_url.startswith("catalogue/"):
        book_url = "https://books.toscrape.com/" + book_url
    
    details = extract_book_details(book_url)
    all_details.append(details)

# Combine original CSV with new details
details_df = pd.DataFrame(all_details)
final_df = pd.concat([books_df, details_df], axis=1)

# Save the full dataset
final_df.to_csv("books_detailed.csv", index=False)
print("Saved all book details to books_detailed.csv")


Scraping books:   0%|          | 0/75 [00:00<?, ?it/s]c:\Users\sboub\anaconda3\Lib\site-packages\soupsieve\css_parser.py:856: FutureWarning: The pseudo class ':contains' is deprecated, ':-soup-contains' should be used moving forward.
  warnings.warn(  # noqa: B028
Scraping books:   1%|▏         | 1/75 [00:01<01:23,  1.13s/it]

Failed to fetch https://books.toscrape.com/the-four-agreements-a-practical-guide-to-personal-freedom_970/index.html: 404 Client Error: Not Found for url: https://books.toscrape.com/the-four-agreements-a-practical-guide-to-personal-freedom_970/index.html


Scraping books:   3%|▎         | 2/75 [00:02<01:16,  1.05s/it]

Failed to fetch https://books.toscrape.com/sophies-world_966/index.html: 404 Client Error: Not Found for url: https://books.toscrape.com/sophies-world_966/index.html


Scraping books:   4%|▍         | 3/75 [00:03<01:14,  1.03s/it]

...

Failed to fetch https://books.toscrape.com/the-edge-of-reason-bridget-jones-2_63/index.html: 404 Client Error: Not Found for url: https://books.toscrape.com/the-edge-of-reason-bridget-jones-2_63/index.html


Scraping books:  99%|█████████▊| 74/75 [01:10<00:00,  1.07it/s]

Failed to fetch https://books.toscrape.com/a-spys-devotion-the-regency-spies-of-london-1_3/index.html: 404 Client Error: Not Found for url: https://books.toscrape.com/a-spys-devotion-the-regency-spies-of-london-1_3/index.html


Scraping books: 100%|██████████| 75/75 [01:11<00:00,  1.05it/s]

Saved all book details to books_detailed.csv


In [27]:
final_df.head()

,Title,Price (£),Rating,Availability,Link,UPC,Description,Genre,Availability (Amount),Number of Reviews
0,Set Me Free,17.46,5,In stock,https://books.toscrape.com/catalogue/set-me-fr...,ce6396b0f23f6ecc,Aaron Ledbetterâs future had been planned ou...,Young Adult,In stock (19 available),0
1,The Four Agreements: A Practical Guide to Pers...,17.66,5,In stock,https://books.toscrape.com/the-four-agreements...,None,None,None,None,None
2,Sophie's World,15.94,5,In stock,https://books.toscrape.com/sophies-world_966/i...,None,None,None,None,None
3,Untitled Collection: Sabbath Poems 2014,14.27,4,In stock,https://books.toscrape.com/untitled-collection...,None,None,None,None,None
4,This One Summer,19.49,4,In stock,https://books.toscrape.com/this-one-summer_947...,None,None,None,None,None


In [24]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re

# Load previously scraped books CSV
books_df = pd.read_csv("books.csv")

def extract_book_details(url):
    """Extract UPC, description, genre, availability, and number of reviews from a book page"""
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Extract product table info
        table_rows = soup.select("table.table.table-striped tr")
        details = {}
        for row in table_rows:
            header = row.select_one("th").text.strip()
            value = row.select_one("td").text.strip()
            if header == "UPC":
                details["UPC"] = value
            elif header == "Availability":
                details["Availability_Details"] = value
            elif header == "Number of reviews":
                details["Number_of_Reviews"] = int(value)
        
        # Extract description
        desc_elem = soup.select_one("#product_description ~ p")
        description = desc_elem.text.strip() if desc_elem else ""
        details["Description"] = description
        
        # Extract genre (breadcrumb)
        breadcrumb = soup.select("ul.breadcrumb li a")
        genre = breadcrumb[2].text.strip() if len(breadcrumb) >= 3 else ""
        details["Genre"] = genre
        
        return details
    except requests.RequestException as e:
        print(f"Failed to fetch {url}: {e}")
        return {"UPC": "", "Availability_Details": "", "Number_of_Reviews": 0, "Description": "", "Genre": ""}

# Loop through each book and extract details
all_details = []
for idx, row in books_df.iterrows():
    print(f"Processing {idx+1}/{len(books_df)}: {row['Title']}")
    book_url = row['Link']
    details = extract_book_details(book_url)
    all_details.append(details)

# Combine with original DataFrame
details_df = pd.DataFrame(all_details)
final_df = pd.concat([books_df, details_df], axis=1)

# Save to CSV
final_df.to_csv("books_detailed.csv", index=False)
print("Saved detailed book data to books_detailed.csv")
print("shape", final_df.shape[0])
final_df.head()

Processing 1/75: Set Me Free
Processing 2/75: The Four Agreements: A Practical Guide to Personal Freedom
Failed to fetch https://books.toscrape.com/the-four-agreements-a-practical-guide-to-personal-freedom_970/index.html: 404 Client Error: Not Found for url: https://books.toscrape.com/the-four-agreements-a-practical-guide-to-personal-freedom_970/index.html
Processing 3/75: Sophie's World
Failed to fetch https://books.toscrape.com/sophies-world_966/index.html: 404 Client Error: Not Found for url: https://books.toscrape.com/sophies-world_966/index.html
Processing 4/75: Untitled Collection: Sabbath Poems 2014
Failed to fetch https://books.toscrape.com/untitled-collection-sabbath-poems-2014_953/index.html: 404 Client Error: Not Found for url: https://books.toscrape.com/untitled-collection-sabbath-poems-2014_953/index.html
Processing 5/75: This One Summer
Failed to fetch https://books.toscrape.com/this-one-summer_947/index.html: 404 Client Error: Not Found for url: https://books.toscrape.co

,Title,Price (£),Rating,Availability,Link,UPC,Availability_Details,Number_of_Reviews,Description,Genre
0,Set Me Free,17.46,5,In stock,https://books.toscrape.com/catalogue/set-me-fr...,ce6396b0f23f6ecc,In stock (19 available),0,Aaron Ledbetterâs future had been planned ou...,Young Adult
1,The Four Agreements: A Practical Guide to Pers...,17.66,5,In stock,https://books.toscrape.com/the-four-agreements...,,,0,,
2,Sophie's World,15.94,5,In stock,https://books.toscrape.com/sophies-world_966/i...,,,0,,
3,Untitled Collection: Sabbath Poems 2014,14.27,4,In stock,https://books.toscrape.com/untitled-collection...,,,0,,
4,This One Summer,19.49,4,In stock,https://books.toscrape.com/this-one-summer_947...,,,0,,


In [23]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import re

BASE_URL = "https://books.toscrape.com/"

def rating_to_int(rating_class):
    """Convert rating text to integer"""
    ratings = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    return ratings.get(rating_class, 0)

def extract_price(price_text):
    """Extract numeric price from text like '£51.77'"""
    match = re.search(r"£([0-9]+\.[0-9]+)", price_text)
    if match:
        return float(match.group(1))
    return 0.0

def scrape_books(limit=10, min_rating=4, max_price=20):
    books = []
    url = BASE_URL
    count = 0
    
    while url and count < limit:
        try:
            response = requests.get(url)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")
            
            for li in soup.select("ol.row li"):
                if count >= limit:
                    break
                
                title_elem = li.select_one("h3 a")
                price_elem = li.select_one(".product_price .price_color")
                availability_elem = li.select_one(".product_price .instock.availability")
                rating_elem = li.select_one("p.star-rating")
                
                if not all([title_elem, price_elem, availability_elem, rating_elem]):
                    continue
                
                title = title_elem["title"].strip()
                price = extract_price(price_elem.text.strip())
                availability = availability_elem.text.strip()
                rating_class = rating_elem.get("class", [])
                rating = rating_to_int(rating_class[1]) if len(rating_class) > 1 else 0
                
                # Filter by rating and price
                if rating < min_rating or price > max_price:
                    continue
                
                product_link = urljoin(BASE_URL, title_elem.get("href"))
                
                books.append({
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating,
                    "Availability": availability,
                    "Link": product_link
                })
                
                count += 1
                # Show progress
                print(f"Scraped {count}/{limit}: {title}")
            
            # Next page
            next_page = soup.select_one("li.next a")
            if next_page:
                next_url = next_page.get("href")
                url = urljoin(url, next_url)
            else:
                url = None
                
        except requests.RequestException as e:
            print(f"Request failed: {e}")
            break
    
    return pd.DataFrame(books)

# Example usage: scrape 10 books with rating >= 4 and price <= 20
if __name__ == "__main__":
    df = scrape_books(limit=100, min_rating=4, max_price=20)
    df.to_csv("books.csv", index=False)
    print("Saved data to books.csv")
print("shape", df.shape)
df.head(df.shape[0])

Scraped 1/100: Set Me Free
Scraped 2/100: The Four Agreements: A Practical Guide to Personal Freedom
Scraped 3/100: Sophie's World
Scraped 4/100: Untitled Collection: Sabbath Poems 2014
Scraped 5/100: This One Summer
Scraped 6/100: Thirst
Scraped 7/100: Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Princess Jellyfish 2-in-1 Omnibus #1)
Scraped 8/100: Princess Between Worlds (Wide-Awake Princess #5)
Scraped 9/100: Outcast, Vol. 1: A Darkness Surrounds Him (Outcast #1)
Scraped 10/100: Mama Tried: Traditional Italian Cooking for the Screwed, Crude, Vegan, and Tattooed
Scraped 11/100: First and First (Five Boroughs #3)
Scraped 12/100: Camp Midnight
Scraped 13/100: The Third Wave: An Entrepreneurâs Vision of the Future
Scraped 14/100: The Stranger
Scraped 15/100: Something More Than This
Scraped 16/100: Poems That Make Grown Women Cry
Scraped 17/100: Dark Notes
Scraped 18/100: Batman: The Dark Knight Returns (Batman)
Scraped 19/100: Agnostic: A Spirited Manifesto
Scraped 20/100: Walt Disney

,Title,Price (£),Rating,Availability,Link
0,Set Me Free,17.46,5,In stock,https://books.toscrape.com/catalogue/set-me-fr...
1,The Four Agreements: A Practical Guide to Pers...,17.66,5,In stock,https://books.toscrape.com/the-four-agreements...
2,Sophie's World,15.94,5,In stock,https://books.toscrape.com/sophies-world_966/i...
3,Untitled Collection: Sabbath Poems 2014,14.27,4,In stock,https://books.toscrape.com/untitled-collection...
4,This One Summer,19.49,4,In stock,https://books.toscrape.com/this-one-summer_947...
...,...,...,...,...,...
70,The Zombie Room,19.69,5,In stock,https://books.toscrape.com/the-zombie-room_87/...
71,The Silent Wife,12.34,5,In stock,https://books.toscrape.com/the-silent-wife_83/...
72,The Girl You Lost,12.29,5,In stock,https://books.toscrape.com/the-girl-you-lost_6...
73,The Edge of Reason (Bridget Jones #2),19.18,4,In stock,https://books.toscrape.com/the-edge-of-reason-...


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import re

BASE_URL = "https://books.toscrape.com/"

def rating_to_int(rating_class):
    #Convert rating text to integer
    ratings = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    return ratings.get(rating_class, 0)

def extract_price(price_text):
    #Extract numeric price from text like '£51.77
    match = re.search(r"£([0-9]+\.[0-9]+)", price_text)
    if match:
        return float(match.group(1))
    return 0.0

def scrape_books(limit=10, min_rating=4, max_price=20):
    books = []
    url = BASE_URL
    count = 0
    
    while url and count < limit:
        try:
            response = requests.get(url)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")
            
            for li in soup.select("ol.row li"):
                if count >= limit:
                    break
                
                title_elem = li.select_one("h3 a")
                price_elem = li.select_one(".product_price .price_color")
                availability_elem = li.select_one(".product_price .instock.availability")
                rating_elem = li.select_one("p.star-rating")
                
                if not all([title_elem, price_elem, availability_elem, rating_elem]):
                    continue
                
                title = title_elem["title"].strip()
                price = extract_price(price_elem.text.strip())
                availability = availability_elem.text.strip()
                rating_class = rating_elem.get("class", [])
                rating = rating_to_int(rating_class[1]) if len(rating_class) > 1 else 0
                
                # Filter by rating and price
                if rating < min_rating or price > max_price:
                    continue
                
                product_link = urljoin(BASE_URL, title_elem.get("href"))
                
                books.append({
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating,
                    "Availability": availability,
                    "Link": product_link
                })
                
                count += 1
            
            # Next page
            next_page = soup.select_one("li.next a")
            if next_page:
                next_url = next_page.get("href")
                url = urljoin(url, next_url)
            else:
                url = None
                
        except requests.RequestException as e:
            print(f"Request failed: {e}")
            break
    
    return pd.DataFrame(books)

# Example usage: scrape 10 books with rating >= 4 and price <= 20
if __name__ == "__main__":
    df = scrape_books(limit=10, min_rating=4, max_price=20)
    df.to_csv("test.csv", index=False)
    print("Saved data to test.csv")
    print("shape", df.shape)
df.head(10)

Saved filtered data to test.csv
shape (10, 5)


,Title,Price (£),Rating,Availability,Link
0,Set Me Free,17.46,5,In stock,https://books.toscrape.com/catalogue/set-me-fr...
1,The Four Agreements: A Practical Guide to Pers...,17.66,5,In stock,https://books.toscrape.com/the-four-agreements...
2,Sophie's World,15.94,5,In stock,https://books.toscrape.com/sophies-world_966/i...
3,Untitled Collection: Sabbath Poems 2014,14.27,4,In stock,https://books.toscrape.com/untitled-collection...
4,This One Summer,19.49,4,In stock,https://books.toscrape.com/this-one-summer_947...
5,Thirst,17.27,5,In stock,https://books.toscrape.com/thirst_946/index.html
6,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5,In stock,https://books.toscrape.com/princess-jellyfish-...
7,Princess Between Worlds (Wide-Awake Princess #5),13.34,5,In stock,https://books.toscrape.com/princess-between-wo...
8,"Outcast, Vol. 1: A Darkness Surrounds Him (Out...",15.44,4,In stock,https://books.toscrape.com/outcast-vol-1-a-dar...
9,Mama Tried: Traditional Italian Cooking for th...,14.02,4,In stock,https://books.toscrape.com/mama-tried-traditio...


In [18]:
df.head()

,Title,Price (£),Rating,Availability,Link
0,Set Me Free,17.46,5,In stock,https://books.toscrape.com/catalogue/set-me-fr...
1,The Four Agreements: A Practical Guide to Pers...,17.66,5,In stock,https://books.toscrape.com/the-four-agreements...
2,Sophie's World,15.94,5,In stock,https://books.toscrape.com/sophies-world_966/i...
3,Untitled Collection: Sabbath Poems 2014,14.27,4,In stock,https://books.toscrape.com/untitled-collection...
4,This One Summer,19.49,4,In stock,https://books.toscrape.com/this-one-summer_947...


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import re

BASE_URL = "https://books.toscrape.com/"

def rating_to_int(rating_class):
    #Convert rating text to integer
    ratings = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    return ratings.get(rating_class, 0)

def extract_price(price_text):
    #Extract numeric price from text like '£51.77
    match = re.search(r"£([0-9]+\.[0-9]+)", price_text)
    if match:
        return float(match.group(1))
    return 0.0

def scrape_books(limit=10):
    books = []
    url = BASE_URL
    count = 0
    
    while url and count < limit:
        try:
            response = requests.get(url)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")
            
            for li in soup.select("ol.row li"):
                if count >= limit:
                    break
                
                title_elem = li.select_one("h3 a")
                price_elem = li.select_one(".product_price .price_color")
                availability_elem = li.select_one(".product_price .instock.availability")
                rating_elem = li.select_one("p.star-rating")
                
                if not all([title_elem, price_elem, availability_elem, rating_elem]):
                    continue
                
                title = title_elem["title"].strip()
                price = extract_price(price_elem.text.strip())
                availability = availability_elem.text.strip()
                rating_class = rating_elem.get("class", [])
                rating = rating_to_int(rating_class[1]) if len(rating_class) > 1 else 0
                
                # Absolute URL of the product
                product_link = urljoin(BASE_URL, title_elem.get("href"))
                
                books.append({
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating,
                    "Availability": availability,
                    "Link": product_link
                })
                
                count += 1
            
            # Handle next page URL
            next_page = soup.select_one("li.next a")
            if next_page:
                next_url = next_page.get("href")
                url = urljoin(url, next_url)
            else:
                url = None
                
        except requests.RequestException as e:
            print(f"Request failed: {e}")
            break
    
    return pd.DataFrame(books)

# Example usage: scrape 10 books and save to CSV
if __name__ == "__main__":
    df = scrape_books(limit=10)
    df.to_csv("test.csv", index=False)
    print("Saved scraped data to test.csv")


Saved scraped data to test.csv


In [15]:
df.head()

,Title,Price (£),Rating,Availability,Link
0,A Light in the Attic,51.77,3,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,1,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,1,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,4,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,https://books.toscrape.com/catalogue/sapiens-a...


In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import re

BASE_URL = "https://books.toscrape.com/"

def rating_to_int(rating_class):
    """Convert rating text to integer"""
    ratings = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    return ratings.get(rating_class, 0)

def extract_price(price_text):
    """Extract numeric price from text like '£51.77'"""
    match = re.search(r"£([0-9]+\.[0-9]+)", price_text)
    if match:
        return float(match.group(1))
    return 0.0

def scrape_books(limit=10):
    books = []
    url = BASE_URL
    count = 0
    
    while url and count < limit:
        try:
            response = requests.get(url)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")
            
            for li in soup.select("ol.row li"):
                if count >= limit:
                    break
                
                title_elem = li.select_one("h3 a")
                price_elem = li.select_one(".product_price .price_color")
                availability_elem = li.select_one(".product_price .instock.availability")
                rating_elem = li.select_one("p.star-rating")
                
                if not all([title_elem, price_elem, availability_elem, rating_elem]):
                    continue
                
                title = title_elem["title"].strip()
                price = extract_price(price_elem.text.strip())
                availability = availability_elem.text.strip()
                rating_class = rating_elem.get("class", [])
                rating = rating_to_int(rating_class[1]) if len(rating_class) > 1 else 0
                
                books.append({
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating,
                    "Availability": availability
                })
                
                count += 1
            
            # Handle next page URL
            next_page = soup.select_one("li.next a")
            if next_page:
                next_url = next_page.get("href")
                url = urljoin(url, next_url)
            else:
                url = None
                
        except requests.RequestException as e:
            print(f"Request failed: {e}")
            break
    
    return pd.DataFrame(books)

# Example usage: scrape 10 books and save to CSV
if __name__ == "__main__":
    df = scrape_books(limit=10)
    df.to_csv("test.csv", index=False)
    print("Saved scraped data to test.csv")


Saved scraped data to test.csv


In [13]:
df.head()

,Title,Price (£),Rating,Availability
0,A Light in the Attic,51.77,3,In stock
1,Tipping the Velvet,53.74,1,In stock
2,Soumission,50.10,1,In stock
3,Sharp Objects,47.82,4,In stock
4,Sapiens: A Brief History of Humankind,54.23,5,In stock


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

# Map textual ratings to numeric values
RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

def scrape_books(min_rating=4, max_price=20):
    """
    Scrape books from 'Books to Scrape' with a minimum rating and maximum price.
    
    Parameters:
    min_rating (int): Minimum rating of the book (1-5)
    max_price (float): Maximum price in £

    Returns:
    pandas.DataFrame: DataFrame containing UPC, Title, Price, Rating, Genre, Availability, Description
    """
    
    base_url = "http://books.toscrape.com/catalogue/page-{}.html"
    all_books = []
    page = 1
    
    while True:
        url = base_url.format(page)
        response = requests.get(url)
        if response.status_code != 200:
            break
        
        soup = BeautifulSoup(response.text, "html.parser")
        books = soup.find_all("article", class_="product_pod")
        if not books:
            break
        
        for book in books:
            # Book title
            title = book.h3.a["title"]
            
            # Price (robust cleaning to handle weird characters)
            price_text = book.find("p", class_="price_color").text
            price_clean = re.sub(r"[^0-9.]", "", price_text)  # Keep digits and dot
            price = float(price_clean)
            
            # Rating
            rating_class = book.p["class"][1]
            rating = RATING_MAP.get(rating_class, 0)
            
            # Availability
            availability = book.find("p", class_="instock availability").text.strip()
            
            # Visit book detail page for UPC, Description, and Genre
            book_url = "http://books.toscrape.com/catalogue/" + book.h3.a["href"].replace("../", "")
            book_resp = requests.get(book_url)
            book_soup = BeautifulSoup(book_resp.text, "html.parser")
            
            # UPC
            upc_tag = book_soup.find("th", text="UPC")
            upc = upc_tag.find_next_sibling("td").text if upc_tag else "Unknown"
            
            # Description
            desc_tag = book_soup.find("meta", attrs={"name": "description"})
            description = desc_tag["content"].strip() if desc_tag else ""
            
            # Genre from breadcrumb
            breadcrumb_items = book_soup.find("ul", class_="breadcrumb").find_all("li")
            genre = breadcrumb_items[2].text.strip() if len(breadcrumb_items) >= 3 else "Unknown"
            
            # Filter by rating and price
            if rating >= min_rating and price <= max_price:
                all_books.append({
                    "UPC": upc,
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating,
                    "Genre": genre,
                    "Availability": availability,
                    "Description": description
                })
        
        page += 1
    
    return pd.DataFrame(all_books)

# Example usage: scrape books with rating >=4 and price <= £20
df = scrape_books(min_rating=4, max_price=20)
print(df.head())


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

# Map textual ratings to numeric values
RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

def scrape_books(min_rating=4, max_price=20):
    """
    Scrape books from 'Books to Scrape' with a minimum rating and maximum price.
    
    Parameters:
    min_rating (int): Minimum rating of the book (1-5)
    max_price (float): Maximum price in £

    Returns:
    pandas.DataFrame: DataFrame containing UPC, Title, Price, Rating, Genre, Availability, Description
    """
    
    base_url = "http://books.toscrape.com/catalogue/page-{}.html"
    all_books = []
    page = 1
    
    while True:
        url = base_url.format(page)
        response = requests.get(url)
        if response.status_code != 200:
            break
        
        soup = BeautifulSoup(response.text, "html.parser")
        books = soup.find_all("article", class_="product_pod")
        if not books:
            break
        
        for book in books:
            # Book title
            title = book.h3.a["title"]
            
            # Price (robust cleaning to handle weird characters)
            price_text = book.find("p", class_="price_color").text
            price_clean = re.sub(r"[^0-9.]", "", price_text)  # Keep digits and dot
            price = float(price_clean)
            
            # Rating
            rating_class = book.p["class"][1]
            rating = RATING_MAP.get(rating_class, 0)
            
            # Availability
            availability = book.find("p", class_="instock availability").text.strip()
            
            # Visit book detail page for UPC, Description, and Genre
            book_url = "http://books.toscrape.com/catalogue/" + book.h3.a["href"].replace("../", "")
            book_resp = requests.get(book_url)
            book_soup = BeautifulSoup(book_resp.text, "html.parser")
            
            # UPC (fixed to avoid deprecation warning)
            upc_tag = book_soup.find("th", string="UPC")
            upc = upc_tag.find_next_sibling("td").text if upc_tag else "Unknown"
            
            # Description
            desc_tag = book_soup.find("meta", attrs={"name": "description"})
            description = desc_tag["content"].strip() if desc_tag else ""
            
            # Genre from breadcrumb
            breadcrumb_items = book_soup.find("ul", class_="breadcrumb").find_all("li")
            genre = breadcrumb_items[2].text.strip() if len(breadcrumb_items) >= 3 else "Unknown"
            
            # Filter by rating and price
            if rating >= min_rating and price <= max_price:
                all_books.append({
                    "UPC": upc,
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating,
                    "Genre": genre,
                    "Availability": availability,
                    "Description": description
                })
        
        page += 1
    
    return pd.DataFrame(all_books)

# Example usage: scrape books with rating >= 4 and price <= £20
df = scrape_books(min_rating=4, max_price=20)
print(df.head())